In [18]:
import pandas as pd


files = [
    "Cherkasy_commercial.csv", "Chernihiv_commercial.csv", "Chernivtsi_commercial.csv", "Dnipro_commercial.csv", "Ivano-Frankivsk_commercial.csv",
    "Kharkiv_commercial.csv", "Khmelnytskyi_commercial.csv", "Kropyvnytskyi_commercial.csv",
    "Lutsk_commercial.csv", "Lviv_commercial.csv", "Mykolaiv_commercial.csv", "Odesa_commercial.csv", "Poltava_commercial.csv", "Rivne_commercial.csv",
    "Sumy_commercial.csv", "Ternopil_commercial.csv", "Uzhhorod_commercial.csv", "Vinnytsia_commercial.csv", "Zaporizhzhia_commercial.csv",
    "Zhytomyr_commercial.csv"
]

dfs = []
for filename in files:
    df = pd.read_csv(filename)
    dfs.append(df)
    print(f"✓ {filename}")

df_ukraine = pd.concat(dfs, ignore_index=True)

df_kyiv = pd.read_csv("kyiv_commercial.csv")

✓ Cherkasy_commercial.csv
✓ Chernihiv_commercial.csv
✓ Chernivtsi_commercial.csv
✓ Dnipro_commercial.csv
✓ Ivano-Frankivsk_commercial.csv
✓ Kharkiv_commercial.csv
✓ Khmelnytskyi_commercial.csv
✓ Kropyvnytskyi_commercial.csv
✓ Lutsk_commercial.csv
✓ Lviv_commercial.csv
✓ Mykolaiv_commercial.csv
✓ Odesa_commercial.csv
✓ Poltava_commercial.csv
✓ Rivne_commercial.csv
✓ Sumy_commercial.csv
✓ Ternopil_commercial.csv
✓ Uzhhorod_commercial.csv
✓ Vinnytsia_commercial.csv
✓ Zaporizhzhia_commercial.csv
✓ Zhytomyr_commercial.csv


In [19]:
df_ukraine.isna().sum()

area                  0
house_type          702
year_of_building    631
wall_type           789
ceiling_height      809
heating             754
is_bank               0
is_office             0
is_services           0
is_warehouse          0
is_production         0
is_free               0
is_retail             0
is_garage             0
is_parking_spot       0
price                18
url                   0
district              0
city                  0
geo_region            0
dtype: int64

In [20]:
df_kyiv.isna().sum()

area                  0
house_type          290
year_of_building    243
wall_type           286
ceiling_height      288
heating             280
is_bank               0
is_office             0
is_services           0
is_warehouse          0
is_production         0
is_free               0
is_retail             0
is_garage             0
is_parking_spot       0
price                 2
url                   0
district              0
city                  0
geo_region            0
dtype: int64

In [21]:
before = len(df_ukraine)
df_ukraine = df_ukraine[df_ukraine["price"].notna()]
df_ukraine = df_ukraine[df_ukraine["price"] > 10]
print(f"Прибрано {before - len(df_ukraine)} рядків (пропущена/аномальна ціна). Лишилось: {len(df_ukraine)}")

Прибрано 23 рядків (пропущена/аномальна ціна). Лишилось: 1572


In [22]:
before = len(df_kyiv)
df_kyiv = df_kyiv[df_kyiv["price"].notna()]
df_kyiv = df_kyiv[df_kyiv["price"] > 10]
print(f"Прибрано {before - len(df_kyiv)} рядків (пропущена/аномальна ціна). Лишилось: {len(df_kyiv)}")

Прибрано 4 рядків (пропущена/аномальна ціна). Лишилось: 2879


In [23]:
def remove_outliers_iqr(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col in cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        mask &= df[col].between(lower, upper)
        print(f"{col}: lower={lower:.1f}, upper={upper:.1f}")
    return df[mask]

before = len(df_ukraine)
df_ukraine = remove_outliers_iqr(df_ukraine, ["price", "area"])
print(f"\nПрибрано {before - len(df_ukraine)} викидів по Україні. Лишилось: {len(df_ukraine)}")

before_kyiv = len(df_kyiv)
df_kyiv = remove_outliers_iqr(df_kyiv, ["price", "area"])
print(f"\nПрибрано {before_kyiv - len(df_kyiv)} викидів в Києві. Лишилось: {len(df_kyiv)}")

price: lower=-855.4, upper=2713.6
area: lower=-273.4, upper=535.6

Прибрано 269 викидів по Україні. Лишилось: 1303
price: lower=-952.8, upper=4977.2
area: lower=-196.7, upper=457.6

Прибрано 474 викидів в Києві. Лишилось: 2405


In [24]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


numeric_targets = ["year_of_building", "ceiling_height"]
purpose_cols = ["is_bank", "is_office", "is_services", "is_warehouse",
                 "is_production", "is_free", "is_retail", "is_garage", "is_parking_spot"]

location_onehot = pd.get_dummies(df_ukraine[["district", "city"]], columns=["district", "city"])

feature_df = pd.concat(
    [df_ukraine[["area", "price"] + numeric_targets + purpose_cols].reset_index(drop=True),
     location_onehot.reset_index(drop=True)],
    axis=1,
)

scaler = StandardScaler()
scaled = scaler.fit_transform(feature_df)

imputer = KNNImputer(n_neighbors=5, weights="distance")
imputed_scaled = imputer.fit_transform(scaled)
imputed = pd.DataFrame(scaler.inverse_transform(imputed_scaled), columns=feature_df.columns, index=df_ukraine.index)

df_ukraine["year_of_building"] = imputed["year_of_building"].round().astype("Int64")
df_ukraine["ceiling_height"] = imputed["ceiling_height"].round(1)

df_ukraine[numeric_targets].isna().sum()

year_of_building    0
ceiling_height      0
dtype: int64

In [25]:
location_onehot = pd.get_dummies(df_kyiv[["district", "city"]], columns=["district", "city"])

feature_df = pd.concat(
    [df_kyiv[["area", "price"] + numeric_targets + purpose_cols].reset_index(drop=True),
     location_onehot.reset_index(drop=True)],
    axis=1,
)

scaled = scaler.fit_transform(feature_df)

imputer = KNNImputer(n_neighbors=5, weights="distance")
imputed_scaled = imputer.fit_transform(scaled)
imputed = pd.DataFrame(scaler.inverse_transform(imputed_scaled), columns=feature_df.columns, index=df_kyiv.index)

df_kyiv["year_of_building"] = imputed["year_of_building"].round().astype("Int64")
df_kyiv["ceiling_height"] = imputed["ceiling_height"].round(1)

df_kyiv[numeric_targets].isna().sum()

year_of_building    0
ceiling_height      0
dtype: int64

In [26]:
for col in ["house_type", "wall_type", "heating"]:
    df_ukraine[col] = df_ukraine[col].fillna("Unknown")

df_ukraine[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [27]:
for col in ["house_type", "wall_type", "heating"]:
    df_kyiv[col] = df_kyiv[col].fillna("Unknown")

df_kyiv[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [28]:
df_ukraine["price"] = df_ukraine["price"].round(0)
df_ukraine["area"] = df_ukraine["area"].round(1)
df_ukraine[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,1303.000000,1303.000000,1303.000000
mean,1034.347659,101.677130,2.801305
std,583.887997,106.902306,0.214898
min,14.000000,2.000000,2.400000
25%,600.000000,24.000000,2.700000
50%,964.000000,63.000000,2.800000
75%,1400.000000,130.000000,2.800000
max,2653.000000,535.000000,4.000000


In [29]:
df_kyiv["price"] = df_kyiv["price"].round(0)
df_kyiv["area"] = df_kyiv["area"].round(1)
df_kyiv[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,2405.000000,2405.000000,2405.000000
mean,2048.755925,113.500042,2.917131
std,1024.539669,95.239168,0.335351
min,13.000000,2.200000,2.000000
25%,1317.000000,42.000000,2.700000
50%,1923.000000,85.000000,2.800000
75%,2667.000000,155.000000,3.000000
max,4967.000000,456.000000,5.000000


In [30]:
df_kyiv.to_csv("Kyiv_commercial_for_analysis.csv", index=False)
df_ukraine.to_csv("Ukraine_commercial_for_analysis.csv", index=False)

In [31]:
df_encoded_ukraine = df_ukraine.drop(columns=["url", "geo_region"])

categorical_cols = ["house_type", "wall_type", "heating", "district", "city"]
df_encoded_ukraine = pd.get_dummies(df_encoded_ukraine, columns=categorical_cols, drop_first=True)
df_encoded_ukraine.to_csv("Ukraine_commercial_ML.csv", index=False)

In [32]:
df_encoded_kyiv = df_kyiv.drop(columns=["url", "geo_region"])

df_encoded_kyiv = pd.get_dummies(df_encoded_kyiv, columns=categorical_cols, drop_first=True)
df_encoded_kyiv.to_csv("Kyiv_commercial_ML.csv", index=False)